# Track 1 — Legacy Well Log Digitisation

Optional inspection and validation starter. **Not** an OCR, curve extraction or calibration solution. Use the approved dataset provided by the organisers. The notebook locates it when run from the repository root or this starter-kit directory; otherwise set `DATA_ROOT` to its location.

In [ ]:
from pathlib import Path
import csv
from PIL import Image
from IPython.display import display

# Resolve data from either the repository root or the starter-kit directory.
def find_data_root():
    candidates = [
        Path.cwd() / "data" / "track_1",
        Path.cwd() / "../../data/track_1",
        Path.cwd() / "../data/track_1",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate data/track_1. Run from the repository root or the track starter-kit "
        "directory, or set DATA_ROOT to the approved dataset location."
    )

DATA_ROOT = find_data_root()
METADATA_DIR = DATA_ROOT / "metadata"
TEMPLATE = METADATA_DIR / "target_depths_template.csv"
IMAGE_DIR = DATA_ROOT / "log_images"
EXPECTED_COLUMNS = ["page_id", "well", "depth_m", "curve", "value", "confidence"]
print("Dataset found:", DATA_ROOT.exists())
print("Template found:", TEMPLATE.exists())
print("Images folder found:", IMAGE_DIR.exists())

In [ ]:
targets = []
if TEMPLATE.is_file():
    with TEMPLATE.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        actual_columns = reader.fieldnames or []
        missing = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
        print("Template columns:", actual_columns)
        print("Missing required columns:", missing)
        if not missing:
            targets = list(reader)
            print("Target rows:", len(targets))
            print("Sample target rows:", targets[:3])
else:
    print("No template at the placeholder path. Ask technical support for the approved data root.")

In [ ]:
images = sorted(
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
) if IMAGE_DIR.is_dir() else []
print("Raster image files:", len(images))
if images:
    try:
        with Image.open(images[0]) as image:
            print("First image:", images[0].name, image.size, image.mode)
            display(image.copy())
    except (OSError, ValueError) as exc:
        print("Image could not be previewed:", exc)
else:
    print("No raster images found at this path; supplied PDFs may require separate local processing.")

## Output-contract validator

Once **your own extraction code** has produced a CSV, set `PREDICTIONS` to its location and run this check. It checks structure and target coverage, **not extraction accuracy**.

In [ ]:
PREDICTIONS = Path("outputs/track1_predictions.csv")  # set after implementing your solution
if PREDICTIONS.is_file() and targets:
    with PREDICTIONS.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
        print("Exact output header:", reader.fieldnames == EXPECTED_COLUMNS)
    key = lambda row: (row["page_id"], row["well"], row["depth_m"], row["curve"])
    expected_keys = [key(row) for row in targets]
    output_keys = [key(row) for row in rows]
    print("Row count matches:", len(rows) == len(targets))
    print("Target identifiers match (including duplicates):", sorted(output_keys) == sorted(expected_keys))
else:
    print("Validator skipped: supply your own output CSV and the approved template first.")

## Your implementation

Implement image/PDF handling, pixel-to-depth/value calibration, curve tracing, QC, confidence and an inference entry point. Freeze fitted/calibrated artefacts before submission; ensure your submitted notebook is reproducible. See the Technical Guide and Submission Guide.

## Validate Your Submission (optional)

Set `SUBMISSION_DIR` to your private team repository's `submission/` folder, then run both cells. Checks formatting and required files, not answer quality or solution execution.

In [ ]:
import csv
import json
import math
from pathlib import Path

HEADERS = {
    "1": ("page_id", "well", "depth_m", "curve", "value", "confidence"),
    "2": ("question_id", "answer", "evidence", "confidence"),
}

def validate_structure(folder):
    folder = Path(folder)
    errors = []
    for name in ("README.md", "requirements.txt", "predictions.csv"):
        path = folder / name
        if not path.is_file() or not path.read_text(encoding="utf-8").strip():
            errors.append("Missing or empty required file: " + name)
    notebooks = sorted(folder.glob("*.ipynb"))
    if not notebooks:
        errors.append("At least one Jupyter notebook (.ipynb) is required")
    for notebook in notebooks:
        try:
            data = json.loads(notebook.read_text(encoding="utf-8"))
            if not isinstance(data.get("nbformat"), int) or not isinstance(data.get("cells"), list):
                errors.append("Invalid notebook structure: " + notebook.name)
        except (json.JSONDecodeError, UnicodeDecodeError):
            errors.append("Unreadable notebook: " + notebook.name)
    return {"valid": not errors, "errors": errors, "notebooks": len(notebooks)}

def validate_csv(path, track):
    path = Path(path)
    errors = []
    if not path.is_file():
        return {"valid": False, "errors": ["Predictions CSV not found"], "rows": 0}
    try:
        with path.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            columns = reader.fieldnames or []
            missing = [name for name in HEADERS[track] if name not in columns]
            if missing:
                errors.append("Missing columns: " + ", ".join(missing))
            if len(columns) != len(set(columns)):
                errors.append("Duplicate CSV column headers")
            rows = list(reader)
    except (UnicodeDecodeError, csv.Error) as exc:
        return {"valid": False, "errors": [f"Unreadable CSV: {exc}"], "rows": 0}
    keys = ("page_id", "depth_m", "curve") if track == "1" else ("question_id",)
    seen = set()
    for number, row in enumerate(rows, 2):
        if None in row or any(v is None for v in row.values()):
            errors.append(f"Line {number}: incorrect number of columns")
            continue
        key = tuple((row.get(k) or "").strip() for k in keys)
        if not all(key):
            errors.append(f"Line {number}: missing identifier")
        elif key in seen:
            errors.append(f"Line {number}: duplicate identifier")
        seen.add(key)
        if track == "1":
            for field in ("depth_m", "value"):
                try:
                    value = float(row.get(field, ""))
                    if not math.isfinite(value):
                        errors.append(f"Line {number}: non-finite {field}")
                except (ValueError, TypeError):
                    errors.append(f"Line {number}: invalid {field}")
        if (row.get("confidence") or "").strip():
            try:
                confidence = float(row["confidence"])
                if not math.isfinite(confidence) or not 0 <= confidence <= 1:
                    errors.append(f"Line {number}: confidence must be between 0 and 1")
            except ValueError:
                errors.append(f"Line {number}: invalid confidence")
    if not rows:
        errors.append("CSV has no data rows")
    return {"valid": not errors, "errors": errors, "rows": len(rows)}




In [ ]:
# Example only: replace pipeline-x and adjust for your current working directory.
SUBMISSION_DIR = Path("../nape-aice-2026-pipeline-x/submission")
report = {"structure": validate_structure(SUBMISSION_DIR), "csv": validate_csv(SUBMISSION_DIR / "predictions.csv", "1")}
print(json.dumps(report, indent=2))
print("PASS: format valid" if all(x["valid"] for x in report.values()) else "FAIL: fix errors above")
